## SEF Y MEF

Para cada segundo (fila) se calcula la potencia total (suma de la potencia de todas las frecuencias): P_total = 10 + 20 + 40 + 20 + 10 + ...

Se calcula la potencia acumulada: 
Frec1 = Pot1, Frec2 = Pot1 + Pot2, Frec3 = Pot1 + Pot2 + Pot3, ...


### SEF: primera frecuencia donde la acumulada alcanza el 95% de la potencia total.

Frecuencia donde la potencia acumulada alcanza el 95%. Percentil 95 de la distribución acumulada de la potencia espectral.
 - SEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.95


### MEF: primera frecuencia donde la acumulada alcanza el 50% de la potencia total.

Mediana de la distribución de potencia espectral.
 - MEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.50


Esto se debe calcular sobre potencia lineal, no sobre dB. 

pot_media = (
    df_dsa_ch1_lin[cols_freq].to_numpy(dtype=float) +
    df_dsa_ch2_lin[cols_freq].to_numpy(dtype=float)
) / 2

# Métricas y correlación

## Z-score
Se normalizan las matrices con z-score global antes de compararlas. Se busca comparar patrones relativos y no solo valores absolutos en dB.

Ambas matrices tienen escalas distintas ya que la exportada en el archivo .f_a parece ser fruto de un procesamiento o suavizado realizado en el propio monitor BIS.

Comparación de las figuras tras la normalización de colores/escala común: tras el z-score, los colores representan intensidad relativa dentro de cada matriz, no dB reales.

$$
Z(t,f)=\frac{D(t,f)-\mu_D}{\sigma_D}
$$

Donde: 
 - D(t,f) es el valor de la DSA en el instante \(t\) 
 - la frecuencia \(f\)
 - $\mu_D$ es la media global de la matriz 
 - $\sigma_D$ su desviación típica.
 

## Correlación
Medida estadística que indica cómo dos variables se relacionan entre sí. Suele ser un número que varía entre -1 y +1. Existen 3 tipos:
 - Correlación positiva (+1):  Ambas variables se mueven en la misma dirección. Si una variable aumenta, la otra también aumenta. 
 - Correlación negativa (-1): Las variables se mueven en direcciones opuestas. Si una variable aumenta, la otra disminuye.
 - Sin correlación (0): No existe ningún patrón predecible entre ambas. El comportamiento de una no afecta ni pr.edice el de la otra.


## Métricas
Para calcularlas primero hay que asegurar que ambas matrices tengan las mismas columnas de frecuencia en el mismo orden y en el mismo rango de tiempo.

Como vamos a comparar matrices que están en escalas distintas (se ve cuando buscamos el valor mínimo y máximo de potencia en cada DSA nos salen valores alejados) se normalizan mediante el z-score global. Esto sirve para comparar patrones de similitud espectral.

Después, en cada matriz solo se seleccionan las posiciones donde ambas matrices tienen datos válidos. Una zona en blanco por valores NaN no se compara.

Al aplicar este filtro, ambas matrices se convierten en vectores de valores válidos de cada matriz. Gracias a esto, se pueden hacer cálculos comparando posiciones.

### MAE
Mide error absoluto medio. Se suele referir a los errores en un modelo de predicción, pero en este caso comparan dos matrices DSA.


Donde:
 - i: cada una de las posiciones de los vectores
 - n: nº total de posiciones
 - A y B: son cada una de las matrices (ahora vectores)
 
Interpretación:
 - 

### RMSE

Interpretación:
 - 
 - También mide error, pero penaliza más los errores grandes porque primero eleva al cuadrado.
 - Si RMSE es bastante mayor que MAE, suele indicar que hay algunos errores puntuales fuertes.


### BIAS
Media de la resta para ver hacia donde se dirige el error. Mide si una matriz tiende a estar sistemáticamente por encima o por debajo de la otra.

Interpretación:
 - bias > 0: entonces la matriz B, normalmente el .f_a, tiene valores mayores que la reconstruida.
 - bias < 0: entonces la reconstruida tiene valores mayores que el .f_a.
 
Como trabajamos con el z-score global en ambas matrices, el bias suele salir muy cerca de 0, porque ambas matrices han sido centradas respecto a su media.


### PEARSON
Cuantificar la similitud lineal entre los valores normalizados de ambas matrices DSA.

En este caso, determina si, cuando una zona de la DSA reconstruida sube, también lo hace la zona equivalente del .f_a de forma proporcional o no.

Interpretación: 
 - +1: relación lineal perfecta positiva
 - 0: sin relación lineal clara
 - -1: relación lineal perfecta negativa
 
Si Pearson aumenta después del suavizado, significa que la reconstruida se está pareciendo más al .f_a en términos de variación lineal de intensidad. No significa que sean idénticas, pero sí que los patrones empiezan a variar de forma parecida.

### SPEARMAN
Evaluar la fuerza y dirección de la asociación entre los arrays. Evaluar si la estructura de intensidades se mantenía entre matrices aunque la relación no fuera estrictamente lineal.